In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from sklearn.metrics import classification_report
from pathlib import Path
np.random.seed(42)


In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "outputs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
RESULTS_FILE = PROJECT_ROOT / "outputs" / "mlp_results.csv"
LOSS_FILE = PROJECT_ROOT / "outputs" / "mlp_loss.csv"
LOSS_IMAGE = PROJECT_ROOT / "outputs" / "plots" / "mlp_loss.png"
TRAIN_FILE = PROJECT_ROOT / "outputs" / "mlp_train_preprocessed.csv"
TEST_FILE = PROJECT_ROOT / "outputs" / "mlp_test_preprocessed.csv"


In [3]:
train_df = pd.read_csv(TRAIN_FILE)
X_train = train_df.drop(columns='high_stress').values
y_train = train_df['high_stress'].values.reshape(-1, 1)

test_df = pd.read_csv(TEST_FILE)
X_test = test_df.drop(columns='high_stress').values
y_test = test_df['high_stress'].values.reshape(-1, 1)


In [4]:
inp = X_train.shape[1]
q1 = 32
q2 = 16
eta = 0.005

W1 = np.random.randn(inp, q1)
W2 = np.random.randn(q1, q2)
W3 = np.random.randn(q2, 1)

def relu(z):
    return np.maximum(0, z)
def sigmoid(z):
    return 1 / (1 + np.exp(-z))
def f(x):
    h1 = relu(W1.T.dot(x))
    h2 = relu(W2.T.dot(h1))
    return sigmoid(W3.T.dot(h2))

In [28]:
errors = []
epochs = 600
n = X_train.shape[0]
for epoch in range(epochs):
    dW1, dW2, dW3 = 0, 0, 0
    for i in range(n):
        x = np.reshape(X_train[i], (inp, 1))
        h1 = relu(W1.T.dot(x))
        h2 = relu(W2.T.dot(h1))
        mat1 = np.heaviside(h1, 0)
        mat2 = np.heaviside(h2, 0)
        err = f(x) - y_train[i]
        dW3 += (1/n) * err * h2
        dW2 += (1/n) * err * np.kron(h1, (W3 * mat2).T)
        dW1 += (1/n) * err * np.kron(x, (W2.dot(W3 * mat2) * mat1).T)
    W3 -= eta * dW3
    W2 -= eta * dW2
    W1 -= eta * dW1

    y_preds = f(X_train.T)
    e = (1/n) * np.sum(-y_train * np.log(y_preds) - (1 - y_train) * np.log(1 - y_preds))
    errors.append(e)

In [29]:
y_test_probs= f(X_test.T)
print(y_test_probs)

y_test_pred = (y_test_probs >= 0.5).astype(int)
accuracy = np.mean(y_test_pred.flatten() == y_test.flatten())
print(accuracy)

[[3.50380377e-04 4.94322219e-02 2.55453741e-02 ... 9.06182324e-01
  2.68264298e-02 5.23260021e-01]]
0.8621929298981426


In [30]:
print(classification_report(y_test.flatten(), y_test_pred.flatten()))

              precision    recall  f1-score   support

           0       0.96      0.85      0.90      2470
           1       0.68      0.91      0.77       868

    accuracy                           0.86      3338
   macro avg       0.82      0.88      0.84      3338
weighted avg       0.89      0.86      0.87      3338



In [32]:
print(W1)
print(W2)
print(W3)

[[ 0.52005615 -0.16765203  0.60038219 ... -0.31206076 -0.60294373
   1.98046324]
 [-0.02082223 -1.05895826  0.82858558 ... -0.18443038 -1.10658764
  -1.19908526]
 [ 0.83756277  1.32550931 -0.11683967 ... -0.34600761 -0.39255414
  -1.32816888]
 ...
 [ 0.04721259  0.69346431 -0.43619307 ... -0.33516697 -1.74453289
  -0.68851415]
 [-0.18365694 -0.11495093 -1.48222994 ... -0.40409531  2.03547739
   1.99217554]
 [-0.13586704 -0.61004155  0.6944068  ...  0.7209438  -0.08770934
   1.49485045]]
[[-2.17966132 -1.47973244  0.6821054  -0.04510572  0.65184394  0.06366373
   0.51276668 -0.83168741  0.31769072  1.01673628  1.22282912  2.00933818
   0.25217125  0.25447697 -1.5188956  -0.5109404 ]
 [-0.21533626  1.01656591 -0.61099771  0.78347816  0.02731439 -0.17713915
   0.23192028 -0.39261747 -0.45070176 -1.69629517 -0.24754683 -0.48202324
  -0.51612159  0.40891824  0.67139603  0.23931748]
 [ 0.98786577  0.55051656 -0.29504939 -0.47471393  0.3504981   0.57329281
   1.35349829  1.18207359 -0.6127693

In [33]:
results_df = pd.DataFrame({
    'true_label': y_test.flatten(),
    'pred_prob': y_test_probs.flatten(),
    'pred_label': y_test_pred.flatten()
})
results_df.to_csv(RESULTS_FILE, index=False)

In [34]:
loss_df = pd.DataFrame({'epoch': range(epochs), 'loss': errors})
loss_df.to_csv(LOSS_FILE, index=False)